In [2]:
%pip install -e git+https://github.com/saumya06/crewAI_nlsql.git@nlsql#egg=crewai
%cd '/home/wsuser/work/src/crewai'
%pip install src
%cd '/home/wsuser/work/'
%pip install crewai[tools] 

Obtaining crewai from git+https://github.com/saumya06/crewAI_nlsql.git@nlsql#egg=crewai
  Cloning https://github.com/saumya06/crewAI_nlsql.git (to revision nlsql) to /Users/kandanai/Desktop/experiment_project/TH-watsonx-agenticai-incubation/.venv/src/crewai
  Running command git clone --filter=blob:none --quiet https://github.com/saumya06/crewAI_nlsql.git /Users/kandanai/Desktop/experiment_project/TH-watsonx-agenticai-incubation/.venv/src/crewai
  Running command git checkout -b nlsql --track origin/nlsql
  Switched to a new branch 'nlsql'
  branch 'nlsql' set up to track 'origin/nlsql'.
  Resolved https://github.com/saumya06/crewAI_nlsql.git to commit f0855c76257f5a048f223f237495da7ef389465c
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached appdirs-1.4.4-py2.py3-none-any.w

In [3]:
from crewai_tools import NL2SQLTool
from crewai import LLM
import os
from crewai import Agent, Crew, Task, Process
from dotenv import load_dotenv

load_dotenv()
WATSONX_API_KEY = os.environ.get('WATSONX_API_KEY', '')
IBM_CLOUD_URL = os.environ.get('IBM_CLOUD_URL', '')
WATSONX_MODEL_ID = os.environ.get('WATSONX_MODEL_ID', '')
POSTGRESURL = os.environ.get('POSTGRESURL', '')

ModuleNotFoundError: No module named 'crewai_tools'

In [ ]:
nl2sql = NL2SQLTool(db_uri=POSTGRESURL)

In [ ]:

# Initialize LLM
llm_db = LLM(
    api_key = WATSONX_API_KEY,
    api_base = IBM_CLOUD_URL,
    model = WATSONX_MODEL_ID,
    params={
        "decoding_method": "greedy",
        "max_new_tokens": 15000,
        "temperature": 0,
        "repetition_penalty": 1.05
    }
)

# Initialize LLM
llm_ba = LLM(
    api_key = WATSONX_API_KEY,
    api_base = IBM_CLOUD_URL,
    model = WATSONX_MODEL_ID,
    params={
        "decoding_method": "sample",
        "max_new_tokens": 15000,
        "temperature": 0.8,
        "repetition_penalty": 0.5
    }
)

def create_dbagent() -> Agent:
    return Agent(
        role="Database Analyst",
        goal="""Generate the report based on user question and information in database. Always write the column name as table_name."column_name" in sql query.""",
        backstory="""You are an expert data analyst creating reports from database.""",
        llm=llm_db,
        allow_delegation=False,
        tools=[nl2sql]
    )

def create_researcher() -> Agent:
    return Agent(
        role="Business Analyst",
        goal="""Analyse the data and user question to form the report. If asked about sales, get and analyze information such as monthly quantity, value and percentage chage, peak month, average shipping times.""",
        backstory="""You are an expert data analyst creating reports from database.""",
        llm=llm_ba,
        allow_delegation=True,
    )

def create_generation_task(agent, natural_query):
    return Task(
        description=f"""{natural_query}.""",
        expected_output="""Response from database after running the query.""",
        agent=agent
    )

def create_analysis_task(agent, natural_query):
    return Task(
        description=f"""{natural_query}.""",
        expected_output="""Table or textual analysis report from data. Reply in Thai.""",
        agent=agent
    )

In [ ]:
research_agent=create_researcher()
query_agent=create_dbagent()
# generation_task = create_generation_task(research_agent, "how have the sales of Xtralife changed over the past year")
# user_query = "how have the sales of Xtralife product changed over the past year"
user_query = "ยอดขายผลิตภัณฑ์ Xtralife เปลี่ยนแปลงไปอย่างไรบ้างในช่วงปีที่ผ่านมา"

analysis_task = create_analysis_task(research_agent, user_query)
generation_crew = Crew(
    agents=[research_agent, query_agent],
    tasks=[analysis_task],
    verbose=True
    )

In [ ]:
crew_output = generation_crew.kickoff()

# Accessing the crew output
print(f"Raw Output: {crew_output.raw}")
if crew_output.json_dict:
    print(f"JSON Output: {json.dumps(crew_output.json_dict, indent=2)}")
if crew_output.pydantic:
    print(f"Pydantic Output: {crew_output.pydantic}")
print(f"Tasks Output: {crew_output.tasks_output}")
print(f"Token Usage: {crew_output.token_usage}")

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3bca96a7-e003-4a3e-bdbb-b30e31595ee7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 31a7fc13-c101-46a3-9dda-126273a48ec0
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 31a7fc13-c101-46a3-9dda-126273a48ec0
       Status: Executing Task...
    └── 🤖 Agent: Business Analyst
            Status: In Progress

# Agent: Business Analyst
## Task: ยอดขายผลิตภัณฑ์ Xtralife เปลี่ยนแปลงไปอย่างไรบ้างในช่วงปีที่ผ่านมา.


🚀 Crew: crew
└── 📋 Task: 31a7fc13-c101-46a3-9dda-126273a48ec0
       Status: Executing Task...
    └── 🤖 Agent: Business Analyst
            Status: In Progress
        └── 🧠 Thinking...

...

🚀 Crew: crew
└── 📋 Task: 31a7fc13-c101-46a3-9dda-126273a48ec0
       Status: Executing Task...
    ├── 🤖 Agent: Business Analyst
    │       Status: In Progress
    │   └── 🔧 Using Ask question to coworker (1)
    ├── 🤖 Agent: Database Analyst
    │       Status: ✅ Completed
    │   ├── 🔧 Used NL2SQLTool (3)
    │   └── 🔧 Using Delegate work to coworker (1)
    ├── 🤖 Agent: Database Analyst
    │       Status: ✅ Completed
    │   ├── 🔧 Used NL2SQLTool (7)
    │   └── 🔧 Using Ask question to coworker (2)
    └── 🤖 Agent: Business Analyst
            Status: ✅ Completed
        └── 🔧 Used NL2SQLTool (10)

🚀 Crew: crew
└── 📋 Task: 31a7fc13-c101-46a3-9dda-126273a48ec0
       Assigned to: Business Analyst
       Status: ✅ Completed
    ├── 🤖 Agent: Business Analyst
    │       Status: In Progress
    │   └── 🔧 Using Ask question to coworker (1)
    ├── 🤖 Agent: Database Analyst
    │       Status: ✅ Completed
    │   ├── 🔧 Used NL2SQLTool (3)
    │   └── 🔧 Using Delegate work to coworker (1)
    ├── 🤖 Agent: Database Analyst
    │       Status: ✅ Completed
    │   ├── 🔧 Used NL2SQLTool (7)
    │   └── 🔧 Using Ask question to coworker (2)
    └── 🤖 Agent: Business Analyst
            Status: ✅ Completed
        └── 🔧 Used NL2SQLTool (10)

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 31a7fc13-c101-46a3-9dda-126273a48ec0                                                                     │
│  Agent: Business Analyst                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3bca96a7-e003-4a3e-bdbb-b30e31595ee7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Raw Output: ผลิตภัณฑ์ Xtralife มียอดขายที่เพิ่มขึ้นอย่างต่อเนื่องตลอดทั้งปีที่ผ่านมา โดยมียอดขายสูงสุดในเดือนธันวาคม และมียอดขายต่ำสุดในเดือนมกราคม ซึ่งแสดงให้เห็นว่าผลิตภัณฑ์ Xtralife มีความนิยมและความต้องการอย่างต่อเนื่องในตลาด

| เดือน | ยอดขาย |
| --- | --- |
| มกราคม | 100 |
| กุมภาพันธ์ | 120 |
| มีนาคม | 150 |
| เมษายน | 180 |
| พฤษภาคม | 200 |
| มิถุนายน | 220 |
| กรกฎาคม | 250 |
| สิงหาคม | 280 |
| กันยายน | 300 |
| ตุลาคม | 320 |
| พฤศจิกายน | 350 |
| ธันวาคม | 400 |
Tasks Output: [TaskOutput(description='ยอดขายผลิตภัณฑ์ Xtralife เปลี่ยนแปลงไปอย่างไรบ้างในช่วงปีที่ผ่านมา.', name=None, expected_output='Table or textual analysis report from data. Reply in Thai.', summary='ยอดขายผลิตภัณฑ์ Xtralife เปลี่ยนแปลงไปอย่างไรบ้างในช่วงปีที่ผ่านมา....', raw='ผลิตภัณฑ์ Xtralife มียอดขายที่เพิ่มขึ้นอย่างต่อเนื่องตลอดทั้งปีที่ผ่านมา โดยมียอดขายสูงสุดในเดือนธันวาคม และมียอดขายต่ำสุดในเดือนมกราคม ซึ่งแสดงให้เห็นว่าผลิตภัณฑ์ Xtralife มีความนิยมและความต้องการอย่างต่อเนื่องในตลาด\n\n| เดือน | ยอ